# Validation and Testing Against Previous Versions

To confirm that $\texttt{MPT-Calculator}$ is working as intended, we validate the software by comparing the outputted tensor coefficients against those generated using a previous version for a range of different test scenarios.

These comparisons have been packaged into a simple testing module, located at $\texttt{./Tests/test\_suite.py}$ and contains code to run simulations for:

<ul>
  <li>A non-magnetic sphere</li>
  <li>A thin magnetic disk</li>
  <li>A non-magnetic inhomogeneous bar</li>
  <li>A magnetic irregular key generated from a step file</li>
  <li>An irregular magnetic tetrahedron</li>
</ul>

The corresponding validation standards are stored in the $\texttt{./Tests/Validation\_Standards/}$ folder in the same standard output format as other results from $\texttt{MPT-Calculator}$. For ease of use, a series of test OCC files are included in the $\texttt{OCC\_Geometry/}$ folder which match those used as validation standards.

<b> Additional test cases can be added easily by modifying the example functions in the module. </b>

<b> Agreement between newly generated results and the validation standard results does not imply that the computed MPT tensor coefficients are "correct" with respect to the true MPT for that object; only that $\texttt{MPT-Calculator}$ is producing consistent results. </b>


## An Example Test Function

The general form of a test function is as follows.

1) Generate a set of results for the geometry under test and keep track of the tensor coefficients.
2) Load the appropriate tensor coefficients from the $\texttt{./Tests/Validation\_Standards/}$ folder.
3) Compute the relative error between the test tensor coefficients and the validation tensor coefficients for each frequency. In the example below we use the Frobenius norm.
4) Record the maximum error.
5) Generate and save comparison plots to the $\texttt{./Tests/Test\_Results/}$ folder.
6) Evaluate if the maximum error is less than a predefined tolerance.

The $\texttt{test_sphere()}$ function is included here as an example.


```python
def test_sphere():
    
    # Running Sweep and computing error.
    geometry = 'OCC_test_sphere_prism_32.py'
    test_results = main(geometry=geometry, order=3, use_OCC=True, use_POD=True)
    test_tensors = test_results['TensorArray'] 
    
    validation_filename = r'Tests/Validation_Standards/OCC_sphere_prism_32/al_0.01_mu_1_sig_1e6/1e1-1e8_40_el_22426_ord_3_POD_13_1e-6/Data'
    valdiation_tensors = np.genfromtxt(validation_filename + '/Tensors.csv', dtype=complex, delimiter=', ')
    
    rel_err = np.zeros(len(test_tensors), dtype=complex)
    for ind in range(len(test_tensors)):
        rel_err[ind] = np.linalg.norm((test_tensors[ind, :] - valdiation_tensors[ind, :])) / np.linalg.norm(valdiation_tensors[ind, :])
    max_err = np.max(rel_err)
    
    # Generating Comparison Graphs
    plt.close('all')
    
    plt.figure()
    plt.loglog(test_results['FrequencyArray'], rel_err.real)
    plt.xlabel('$\omega$, [rad/s]')
    plt.ylabel('Relative Error')
    plt.savefig('Tests/Test_Results/Sphere_rel_err.pdf')

    # Plotting overlays of the new and old tensor coefficients.
    plt.figure()
    for i in range(9):
        if i == 0:
            plt.semilogx(test_results['FrequencyArray'], test_results['TensorArray'][:,i].real, label='New', color='b')
            plt.semilogx(test_results['FrequencyArray'], valdiation_tensors[:,i].real, label='Standard', color='r')
        else:
            plt.semilogx(test_results['FrequencyArray'], test_results['TensorArray'][:,i].real, color='b')
            plt.semilogx(test_results['FrequencyArray'], valdiation_tensors[:,i].real, color='r')
    plt.xlabel('$\omega$, [rad/s]')
    plt.ylabel(r'$(\tilde{\mathcal{R}})_{ij}$, [m$^3$]')
    plt.legend()
    plt.savefig('Tests/Test_Results/Sphere_real.pdf')
    
    plt.figure()
    for i in range(9):
        if i == 0:
            plt.semilogx(test_results['FrequencyArray'], test_results['TensorArray'][:,i].imag, label='New', color='b')
            plt.semilogx(test_results['FrequencyArray'], valdiation_tensors[:,i].imag, label='Standard', color='r')
        else:
            plt.semilogx(test_results['FrequencyArray'], test_results['TensorArray'][:,i].imag, color='b')
            plt.semilogx(test_results['FrequencyArray'], valdiation_tensors[:,i].imag, color='r')
    plt.xlabel('$\omega$, [rad/s]')
    plt.ylabel(r'$(\mathcal{I})_{ij}$, [m$^3$]')
    plt.legend()
    plt.savefig('Tests/Test_Results/Sphere_imag.pdf')
    
    plt.close('all')
    
    # Test that maximum error is less than tolerance. The assert keyword raises an error if comparion returns False.
    assert max_err < 1e-2

```

In the above script we are generating plots for the relative error as a function of frequency, which we would typically expect to get larger as $\omega$ increases and the discretisation and POD snapshots become insufficient to fully capture the behavior. In addition, we generate plots for overlays of both the new tensor coefficients and the validation tensor coefficients. This can aid in debugging.


<b> Note that the name of each function is prefixed with "$\texttt{test\_}$". This is used for automatic unit testing.</b> 


## The $\texttt{pytest}$ Testing Library

In our testing, we use pytest as the testing library, with documentation avaliable [here](https://docs.pytest.org/en/8.2.x/). Pytest is a standard unit testing library for Python and, once installed, can be called from the commandline or terminal via the following syntax:
```bash
>> python3 -m pytest -s test_suite.py
```
from the Tests directory. E.g.

 <img src="Figures/cmd_pytest1.png" alt="isolated" width="500"/> 

The $\texttt{pytest}$ module runs each function with the "$\texttt{test\_}$" prefix in the $\texttt{test\_suite.py}$ module. In our published version there are 5 such functions. The $\texttt{pytest}$ command will run each of these functions one after the other and record if they encountered an error (e.g. AssertionError), a warning (e.g. DepreciationWarning), or if the function ran without any issue.

Alternatively, specific individual tests can be run using $\texttt{::}$ as a seperator between the module name ($\texttt{test\_suite.py}$) and the function name which will run only that specific function. e.g.

 <img src="Figures/cmd_pytest3.png" alt="isolated" width="500"/>  <img src="Figures/cmd_pytest4.png" alt="isolated" width="500"/>

Once the code has finished running, we can inspect any saved figures, such as those saved in the $\texttt{./Tests/Test\_Results/}$ folder.
